In [1]:
import os
import pandas as pd
import numpy as np

np.random.seed(42)

if os.path.exists('upi_txns.csv'):
    upi_df = pd.read_csv('upi_txns.csv')
else:
    upi_df = pd.DataFrame({
        'txn_id': range(1, 501),
        'city': np.random.choice(['Ahmedabad', 'Surat', 'Vadodara', 'Rajkot'], size=500),
        'category': np.random.choice(['Groceries', 'Food', 'Fuel', 'Recharge', 'Rent', 'Shopping'], size=500),
        'bank_app': np.random.choice(['PayEasy', 'QuickPe', 'RupeeGo'], size=500),
        'amount_inr': np.random.randint(20, 5001, size=500),
        'status': np.random.choice(['Success', 'Failed', 'Pending'], size=500, p=[0.90, 0.07, 0.03])
    })

upi_df['hour'] = np.random.randint(6, 24, size=len(upi_df))
upi_df['wait_seconds'] = np.random.randint(1, 61, size=len(upi_df))
upi_df['cashback_inr'] = (upi_df['amount_inr'] * 0.02).round(2)

upi_df.to_csv('upi_eda_input.csv', index=False)
upi_df.head()

,txn_id,city,category,bank_app,amount_inr,status,hour,wait_seconds,cashback_inr
0,1,Vadodara,Rent,QuickPe,3715,Success,12,35,74.30
1,2,Rajkot,Rent,PayEasy,3472,Success,20,19,69.44
2,3,Ahmedabad,Groceries,PayEasy,934,Success,16,20,18.68
3,4,Vadodara,Fuel,QuickPe,1328,Success,13,57,26.56
4,5,Vadodara,Food,QuickPe,4197,Success,12,18,83.94


In [2]:
amount_summary = upi_df[['amount_inr', 'hour', 'wait_seconds']].describe()
status_share = (upi_df['status'].value_counts(normalize=True) * 100).round(1)

print(amount_summary)
print("\nStatus Share (%):")
print(status_share)

        amount_inr        hour  wait_seconds
count   500.000000  500.000000    500.000000
mean   2493.704000   14.088000     29.820000
std    1455.259642    5.316087     17.603732
min      21.000000    6.000000      1.000000
25%    1229.000000    9.000000     14.000000
50%    2483.500000   14.000000     30.000000
75%    3770.250000   19.000000     46.000000
max    4984.000000   23.000000     60.000000

Status Share (%):
status
Success    90.2
Failed      6.2
Pending     3.6
Name: proportion, dtype: float64


In [4]:
upi_df['is_failed'] = (upi_df['status'] == 'Failed').astype(int)

failure_table = (
    upi_df.groupby('bank_app')
    .agg(
        txn_count=('txn_id', 'count'),
        failed_pct=('is_failed', lambda s: round(s.mean() * 100, 1))
    )
    .sort_values(by='failed_pct', ascending=False)
)

failure_table

,txn_count,failed_pct
bank_app,,
QuickPe,164,7.3
PayEasy,160,6.2
RupeeGo,176,5.1


In [3]:
corr_cols = ['amount_inr', 'hour', 'wait_seconds', 'cashback_inr']
corr_matrix = upi_df[corr_cols].corr().round(2)

corr_matrix.to_csv('upi_corr.csv')

strongest_corr_col = corr_matrix['amount_inr'].drop('amount_inr').abs().idxmax()
print("Most strongly correlated with amount_inr:", strongest_corr_col)
corr_matrix

Most strongly correlated with amount_inr: cashback_inr


,amount_inr,hour,wait_seconds,cashback_inr
amount_inr,1.00,-0.01,-0.03,1.00
hour,-0.01,1.00,-0.05,-0.01
wait_seconds,-0.03,-0.05,1.00,-0.03
cashback_inr,1.00,-0.01,-0.03,1.00
